# 02 — Fine-tune the bi-encoder (RD-09)

## Read this before running anything

RD-09 is **held, not open**, and the reasons are measured rather than argued:

| finding | number |
|---|---|
| The existing fine-tune vs. its own base model | **+4.5pp** — *below* the 6pp bar, so a null result |
| Swapping what was indexed instead (RD-02) | **+12.9 to +15.7pp**, no retraining |
| RD-09's corpus bet, run by proxy (RD-16) | **−7.0pp**, the only significant result in that sweep |
| The slice retraining competes for | 16.4% never-retrieved |

That last row is the honest framing. Retraining can only help queries where the
answer is *never retrieved at all*. Reordering the ones that are retrieved but
ranked low is a different problem, and it belongs to notebook `03`.

**None of this means don't try.** It means: know the prior, pre-register the
prediction, and let the number decide. The fine-tune is also better than its
reputation — against five modern encoders at the same width it wins every one,
and the only thing that beat it needed 5× the parameters for a null-result
+2.8pp. What is weak is the *fine-tuning gain*, not the model.

## The mistake this notebook exists to not repeat

The original run: 181,149 triplets, `MultipleNegativesRankingLoss`, 3 epochs,
`eval_on_start: False`, `prediction_loss_only: True` — **no evaluator and no
held-out split at all**. Its recorded 10.9% describes memorisation and cannot be
cited for anything.

In [ ]:
import sys, os
from pathlib import Path

# rdlib lives at training/rdlib; this notebook is at training/notebooks.
sys.path.insert(0, str(Path.cwd().parent))

# Cells are large and the darwin default lives under os.tmpdir(), which gets
# reaped. Point this somewhere durable and OUTSIDE the repo -- the working tree
# is in OneDrive, which would try to sync ~170 MB per cell.
os.environ.setdefault("EVAL_CELL_DIR", str(Path.home() / "rd_eval_cells"))

import rdlib
from rdlib import paths
print("repo      ", paths.REPO_ROOT)
print("cells     ", paths.cell_dir())

In [ ]:
from rdlib import parity
parity.report()   # never train against a harness you have not pinned

## Step 1 — pre-register the prediction

METHODS §9 does this for every experiment, and §9a records *why*: revising a
commitment after seeing which way it cuts is exactly the failure a commitment
exists to prevent. Write the prediction now, while it is still a prediction.

In [ ]:
import json, datetime
from pathlib import Path
from rdlib.paths import ARTIFACTS_DIR

EXPERIMENT = "rd22_biencoder_wikt_paraphrase"

PREREGISTRATION = {
    "experiment": EXPERIMENT,
    "date": datetime.date.today().isoformat(),
    "hypothesis": (
        "Training on (Wiktionary gloss -> WordNet gloss) paraphrase pairs teaches "
        "the description-to-definition relation that MS MARCO and QA corpora do "
        "not, and should move lenient R@1 above the fine-tune control."
    ),
    "control": "rd22_gloss_ft (= RD-16 full_gloss_ft, 25.4% lenient R@1)",
    "primary_metric": "lenient R@1, authored-reachable slice (n=287)",
    "decision_rule": "METHODS 9a: under ~6pp is a NULL RESULT, not a win",
    # State these BEFORE the run. An unfalsifiable prediction is not one.
    "predicted_delta_pp": None,     # <-- fill in
    "predicted_echo_change": None,  # <-- fill in
    "what_would_falsify_it": (
        "A delta at or below zero, or a gain that arrives together with an echo "
        "increase -- which would mean it bought recall by reintroducing lexical "
        "overlap, exactly what RD-12's lemma-gloss arm did."
    ),
}

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
prereg_path = ARTIFACTS_DIR / f"{EXPERIMENT}.prereg.json"
prereg_path.write_text(json.dumps(PREREGISTRATION, indent=2))
print(json.dumps(PREREGISTRATION, indent=2))

## Step 2 — build training pairs

Three recipes ship in `rdlib.pairs`. Each is documented with what it teaches and
where it is weak; read the docstrings before choosing.

| recipe | pairs | teaches |
|---|---|---|
| `gloss_to_lemma` | 117,791 | the **original** recipe — a 12-word description against a one-token document. Included as a control |
| `example_to_gloss` | 37,451 | usage sentence → definition. Free, genuinely cross-register, noisy |
| `wiktionary_paraphrase` | ~large | **two independent definitions of the same word.** The paraphrase relation itself |

The third is the one worth trying first: two lexicographers defining the same
word separately produce a genuine paraphrase pair, no language model involved,
so no LLM register leaks in. It needs the Kaikki dump
(`npm run supplement:fetch`, ~3.2 GB).

In [ ]:
from rdlib import pairs as P

# Start with the two that need no download. Add wiktionary_paraphrase once the
# dump is present -- it is a few minutes to stream 3.2 GB.
data = P.pairs_example_to_gloss()
print(P.summarise(data))

# Uncomment when ~/rd_sources/kaikki-english.jsonl exists:
# data = data + P.pairs_wiktionary_paraphrase()
# print(P.summarise(data))

In [ ]:
for p in data[:5]:
    print(f"  [{p.recipe}] {p.target}")
    print(f"     query: {p.query[:88]}")
    print(f"     doc  : {p.doc[:88]}\n")

## Step 3 — the contamination gate

This raises rather than warns. A verbatim eval query in training is direct
contamination and makes every downstream number worthless.

`check_targets` is the stricter setting and is on by default. For a
full-vocabulary bi-encoder there is a defensible case for turning it off — the
eval targets are ordinary English words, and excluding them biases the
vocabulary the model sees. But it must be a **deliberate, recorded** choice, not
a default that got clicked past. If you turn it off, say so in the
pre-registration above.

In [ ]:
from rdlib.evalset import assert_disjoint, load_eval_set

ev = load_eval_set()
pair_tuples = [(p.query, p.target) for p in data]

try:
    assert_disjoint(pair_tuples, ev, check_targets=True)
    print("PASS — no eval query and no eval target appears in training")
except RuntimeError as exc:
    print(exc)

**It will fire.** The eval targets are ordinary English words drawn from the
same WordNet the pairs come from, so any raw slice overlaps. That is the gate
working, not a bug.

Two legitimate responses, and the choice belongs in the pre-registration:

1. **Drop the overlapping pairs** (below). Cleanest, and what a cautious
   experiment does. Costs a little vocabulary coverage.
2. **`check_targets=False`** — defensible for a full-vocabulary bi-encoder,
   since excluding 312 common words biases what the model sees. The *queries*
   are still checked, and those are the actual contamination risk. But record
   the reason; do not just flip it.

In [ ]:
# Response 1: drop pairs whose target is an eval target.
eval_targets = {r.target.lower() for r in ev}
clean = [p for p in data if p.target.lower() not in eval_targets]
print(f"dropped {len(data) - len(clean):,} of {len(data):,} pairs")

assert_disjoint([(p.query, p.target) for p in clean], ev, check_targets=True)
print("PASS")
data = clean

## Step 4 — the split, decided now

Split on the **word**, not the row. Splitting by row would put one definition of
a word in train and another in validation, and the validation number would then
measure memorisation — the original run's mistake, in miniature.

This validation set is for watching the loss and choosing a checkpoint. **It is
not the benchmark.** The benchmark is `eval/sets/v1.jsonl`, scored through
retrieval, and §9a resolves on lenient R@1 there and nowhere else.

In [ ]:
train_pairs, val_pairs = P.split(data, val_fraction=0.05, by_target=True)
train_ds, val_ds = P.to_dataset(train_pairs), P.to_dataset(val_pairs)

overlap = {p.target for p in train_pairs} & {p.target for p in val_pairs}
print(f"train {len(train_pairs):,}   val {len(val_pairs):,}   target overlap {len(overlap)}")
assert not overlap, "split leaked -- targets appear on both sides"

## Step 5 — train

`MultipleNegativesRankingLoss` mines negatives **in-batch**, so batch size *is*
the number of negatives each example sees. That is also why the original run's
training-time figure is meaningless as a retrieval number: ranking against 63
in-batch negatives and ranking against 693,325 live candidates are different
problems, and this whole project exists partly because they disagreed.

Starting from the existing fine-tune continues its training; pass
`BASE_MODEL` instead to start from `all-MiniLM-L6-v2` and reproduce the original
experiment cleanly.

**384 dimensions is a hard constraint if you want to ship.** `GlossEmbedding` is
`halfvec(384)`. A wider model is measurable but unshippable — that is one of the
three independent reasons `all-mpnet-base-v2` was rejected at +2.8pp.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from sentence_transformers import SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer import losses

from rdlib.build import PRODUCTION_MODEL, BASE_MODEL

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
OUT_DIR = ARTIFACTS_DIR / EXPERIMENT

model = SentenceTransformer(PRODUCTION_MODEL, device=DEVICE)
assert model.get_embedding_dimension() == 384, "must stay 384-dim to be shippable"

loss = losses.MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=str(OUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=64,   # = the number of in-batch negatives
    learning_rate=2e-5,
    warmup_ratio=0.1,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    logging_steps=50,
    seed=20260830,
    report_to=[],
)

trainer = SentenceTransformerTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds, loss=loss,
)
trainer.train()
model.save(str(OUT_DIR / "final"))
print(f"saved -> {OUT_DIR / 'final'}")

## Step 6 — score it the way the project scores things

A falling loss is not a result. Build a cell with the new encoder and run it
through the same retrieval path everything else is measured on.

In [ ]:
from rdlib.build import build_wordnet_cell
from rdlib.cells import load_cell
from rdlib.evalset import headline
from rdlib.retrieval import run_eval
from rdlib.build import load_encoder, encode_texts
from rdlib.metrics import score, compare

CELL = f"cell_{EXPERIMENT}"
info = build_wordnet_cell(CELL, model_id=str(OUT_DIR / "final"))
print(info)

rows = headline(ev)
cell = load_cell(CELL)
enc_model = load_encoder(str(OUT_DIR / "final"))
results = run_eval(cell, rows, lambda t: encode_texts(enc_model, t, progress=False))
m = score(results)

print(f"\n  lenient R@1 {m.lenient_recall1*100:5.1f}%   (control 25.4%)")
print(f"  strict  R@1 {m.recall1*100:5.1f}%   (control 21.6%)")
print(f"  R@10        {m.recall10*100:5.1f}%   (control 51.6%)")
print(f"  MRR@10      {m.mrr10:5.3f}    (control 0.304)")
print(f"  echo        {m.echo_rate*100:5.1f}%   (control 14.6%)")

In [ ]:
# The paired test. Comparing two independent R@1 figures at n=287 cannot see a
# three-point change; this can, because it looks only at disagreements.
control_cell = load_cell("rd22_gloss_ft")
control_model = load_encoder("franzclarin/ReverseDictionary")
control = run_eval(control_cell, rows,
                   lambda t: encode_texts(control_model, t, progress=False))

c = compare(control, results, lenient=True)
print(f"delta        {c['delta_pp']:+.1f}pp")
print(f"wins/regr    {c['n_wins']}W / {c['n_regressions']}R")
print(f"p            {c['p']:.4f}")
print(f"9a verdict   {'CLEARS the bar' if c['clears_9a_bar'] else 'NULL RESULT -- do not act on it'}")
print()
print("regressions:", ", ".join(c["regressions"][:12]))

## Step 7 — what to do with the answer

**If it is a null result** (under ~6pp): record it. A negative result recorded
properly is worth more than an unrecorded positive one — RD-12 and RD-16 are
both negative and both saved later work. File it as a ticket with the numbers.

**If it clears the bar:** it is not shipped yet. Go to
`04_export_and_ship_gate.ipynb`. Shipping needs (a) confirmation through
`npx tsx scripts/eval.ts`, not this notebook; (b) a 384-dim ONNX export that
fits RD-11's function bundle; and (c) a full re-embed of all 693,325
`GlossEmbedding` rows plus an IVFFlat rebuild — which needs
`SET maintenance_work_mem` in the migration, or it fails at 64 MB.

**Either way, check echo.** A recall gain that arrives with an echo increase
bought its points by reintroducing lexical overlap. RD-12's lemma-gloss arm did
exactly that, and it is why echo is a primary metric here.